# Notebook 3 – Image Preprocessing



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept → explain → example
PIL/NumPy often store HWC; PyTorch CNNs expect NCHW. `ToTensor` converts uint8 `[0,255]` to float `[0,1]`. Normalization applies `(x-mean)/std`; compute channel statistics on training data only and reuse for validation/test. Resizing can distort aspect ratio and crops can remove evidence. For grayscale, use a one-channel model or deliberately adapt to an RGB pretrained model.

Augmentation belongs in training preprocessing only. Validation/test preprocessing should be deterministic. Some transformations invalidate labels: flipping text, digits, or left/right medical findings is unsafe.

In [ ]:
from PIL import Image, ImageDraw
from torchvision import transforms
import matplotlib.pyplot as plt
image=Image.new('RGB',(120,80),'white'); ImageDraw.Draw(image).ellipse((35,15,85,65),fill=(220,70,50))
resize=transforms.Resize((64,64)); crop=transforms.CenterCrop((48,48))
t=transforms.ToTensor()(crop(resize(image)))
print('PIL:',image.size,image.mode,'tensor:',tuple(t.shape),t.dtype,(t.min().item(),t.max().item()))
mean,std=(.5,.5,.5),(.25,.25,.25)
train_tf=transforms.Compose([transforms.Resize((64,64)),transforms.RandomCrop(56),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.Resize((64,64)),transforms.CenterCrop(56),transforms.ToTensor(),transforms.Normalize(mean,std)])
print('train/eval shapes:',train_tf(image).shape,eval_tf(image).shape)
plt.imshow(t.permute(1,2,0)); plt.axis('off'); plt.show()

PIL: (120, 80) RGB tensor: (3, 48, 48) torch.float32 (0.19607843458652496, 1.0)
train/eval shapes: torch.Size([3, 56, 56]) torch.Size([3, 56, 56])


## Implement and interpret
Compare image before/after transforms, and inspect tensor shape/range. For pretrained networks, use the specific weights' documented resize, crop, and normalization. For training from scratch, estimate mean/std using training images only. Keep class semantics and aspect ratio in mind; do not apply a generic recipe blindly.

**In your own words:** Why are random training crops inappropriate for deterministic test evaluation?